# 06 - Classical baselines (TF-IDF)

Three linear baselines on `clean_text`: logistic regression and a linear SVM on stacked word 1-2 grams + character 3-5 grams, and multinomial naive Bayes on word n-grams. Hyperparameters are fixed in `src/models.py`; the validation partition is used only to sanity-check them, the test partition is scored once.

Each model writes `results/predictions/<model>_test.csv` with `review_id, y_true, y_pred, p_dissat` so the transformer predictions from notebook 07 can be paired against them instance by instance in notebook 08.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# make src importable when the kernel starts inside notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", context="notebook")
from src import config
from src.io import ensure_dirs, save_table, update_metrics, read_metrics
ensure_dirs()

In [2]:
import time
from src.models import build_baselines, positive_score
from src.evaluation import macro_f1_report, per_language_metrics, confusion
from src.io import save_predictions

train = pd.read_parquet(config.SPLIT_FILES["train"])
val = pd.read_parquet(config.SPLIT_FILES["val"])
test = pd.read_parquet(config.SPLIT_FILES["test"])
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})

{'train': 34250, 'val': 7340, 'test': 7340}


In [3]:
models = build_baselines()
results = {}
for key, pipe in models.items():
    t0 = time.time()
    pipe.fit(train.clean_text, train.satisfaction_label)
    val_pred = pipe.predict(val.clean_text)
    test_pred = pipe.predict(test.clean_text)
    score = positive_score(pipe, test.clean_text, config.POSITIVE_CLASS)
    rep_val = macro_f1_report(val.satisfaction_label, val_pred)
    rep = macro_f1_report(test.satisfaction_label, test_pred, score)
    results[key] = rep
    save_predictions(key, pd.DataFrame({"review_id": test.review_id, "y_true": test.satisfaction_label,
                                        "y_pred": test_pred, "p_dissat": score}))
    print(f"{config.MODEL_DISPLAY[key]:36s} val macro-F1 {rep_val['macro_f1']:.3f} | "
          f"test macro-F1 {rep['macro_f1']:.3f}  recall(dissat) {rep['recall_Dissatisfied']:.3f}  "
          f"AUC {rep['auc']:.3f}  ({time.time()-t0:.0f}s)")

TF-IDF + logistic regression         val macro-F1 0.904 | test macro-F1 0.908  recall(dissat) 0.933  AUC 0.960  (12s)


TF-IDF + linear SVM                  val macro-F1 0.906 | test macro-F1 0.911  recall(dissat) 0.944  AUC 0.961  (11s)


TF-IDF + multinomial naive Bayes     val macro-F1 0.887 | test macro-F1 0.890  recall(dissat) 0.943  AUC 0.945  (3s)


In [4]:
summary = pd.DataFrame(results).T.round(4)
summary.index = [config.MODEL_DISPLAY[k] for k in summary.index]
save_table(summary.reset_index().rename(columns={"index": "model"}), "baselines_test")
summary[["n", "macro_f1", "recall_Dissatisfied", "recall_Satisfied", "precision_Dissatisfied", "auc", "accuracy_secondary"]]

,n,macro_f1,recall_Dissatisfied,recall_Satisfied,precision_Dissatisfied,auc,accuracy_secondary
TF-IDF + logistic regression,7340.0,0.9083,0.9326,0.8813,0.9077,0.9605,0.9098
TF-IDF + linear SVM,7340.0,0.9107,0.9441,0.8727,0.9027,0.9605,0.9124
TF-IDF + multinomial naive Bayes,7340.0,0.8904,0.9434,0.8301,0.8741,0.9448,0.8931


## Per-language view

In [5]:
from src.io import load_predictions
preds = load_predictions()
lang_rows = []
for key in models:
    p = preds[key].merge(test[["review_id", "language", "length_band"]], on="review_id")
    pl = per_language_metrics(p, p.y_true, p.y_pred)
    pl.insert(0, "model", config.MODEL_DISPLAY[key])
    lang_rows.append(pl)
by_lang = pd.concat(lang_rows, ignore_index=True).round(4)
save_table(by_lang, "baselines_by_language")
by_lang

,model,language,n,macro_f1,recall_Dissatisfied,recall_Satisfied,accuracy_secondary
0,TF-IDF + logistic regression,Arabic,1781,0.8798,0.8573,0.9014,0.8827
1,TF-IDF + logistic regression,English,5559,0.9145,0.9497,0.8722,0.9185
2,TF-IDF + linear SVM,Arabic,1781,0.8831,0.8758,0.8926,0.8855
3,TF-IDF + linear SVM,English,5559,0.9168,0.9597,0.8637,0.9210
4,TF-IDF + multinomial naive Bayes,Arabic,1781,0.8424,0.7688,0.9072,0.8484
5,TF-IDF + multinomial naive Bayes,English,5559,0.9002,0.9831,0.7948,0.9074


## What the best baseline keys on

Top weighted features of the logistic regression, separately for each class, as a sanity check that the model is learning sentiment rather than app names.

In [6]:
lr = models["tfidf_lr"]
feat_names = lr.named_steps["features"].get_feature_names_out()
coef = lr.named_steps["clf"].coef_[0]
classes = list(lr.named_steps["clf"].classes_)
pos_class = classes[1]     # positive coefficient pushes toward classes[1]
order = np.argsort(coef)
print(f"strongest toward {pos_class}:")
print("  ", ", ".join(feat_names[i].split("__", 1)[1] for i in order[-25:][::-1]))
print(f"strongest toward {classes[0]}:")
print("  ", ", ".join(feat_names[i].split("__", 1)[1] for i in order[:25]))

strongest toward Satisfied:
   ممتاز, great, best, شكر, no need, good, i like, جميل, easy to, رايع, nice, very useful, easy, best app, please add, not showing, amazing, excellent, showing the, جيد جدا, using this, login please, nice app, love, سهل
strongest toward Dissatisfied:
   t, not, can t, bad, worst, لا, not easy, at all, not good, no, very bad, سيء,  no,  un,  سي, version was, not user, سي, not , wors,  wors, غير, ors, useless,  not 


In [7]:
update_metrics("baselines", {k: v for k, v in results.items()})
print("saved predictions:", sorted(p.name for p in config.PREDICTIONS_DIR.glob('*.csv')))

saved predictions: ['tfidf_lr_test.csv', 'tfidf_nb_test.csv', 'tfidf_svm_test.csv']
